# Nettoyage de la base d'apprentissage

In [2]:
import numpy as np
import pandas as pd
import sys
import os

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from cleaning import *

In [3]:
# On charge la base d'apprentissage et celle des classements FIFA
df = pd.read_csv("../data_finale/base_apprentissage.csv")
df_fifa = pd.read_csv("../data/classement_fifa/fifa_ranking_fin_saison.csv", sep=",", encoding="utf-8-sig")

In [4]:
df

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,injury_minor_unknown_nb_d,injury_minor_unknown_nb_m,injury_musculaire,injury_genou,injury_cheville_pied,injury_mollet_tibia,injury_dos_bassin,injury_trauma_severe,injury_medical_repos,injury_minor_unknown
0,ENG-Premier League,2021,Arsenal,Ainsley Maitland-Niles,ENG,"MF,DF",22,1997.0,11,5,...,12.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,ENG-Premier League,2021,Arsenal,Alexandre Lacazette,FRA,FW,29,1991.0,31,22,...,23.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,ENG-Premier League,2021,Arsenal,Bernd Leno,GER,GK,28,1992.0,35,35,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ENG-Premier League,2021,Arsenal,Bukayo Saka,ENG,MF,18,2001.0,32,30,...,4.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,ENG-Premier League,2021,Arsenal,Calum Chambers,ENG,DF,25,1995.0,10,8,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17117,ITA-Serie A,2425,Parma,Mathias Løvik,NOR,"DF,MF",20,2003.0,6,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17118,ITA-Serie A,2425,Venezia,Mirko Marić,CRO,FW,29,1995.0,10,4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17119,ITA-Serie A,2526,Genoa,Albert Grønbaek,DEN,MF,24-346,2001.0,4,1,...,16.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
17120,ITA-Serie A,2526,Parma,Mathias Løvik,NOR,MF,22-149,2003.0,9,5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Traitement des doublons

In [5]:
verifier_doublons_metier_et_techniques(df)

Recherche de doublons
 Attention : 787 lignes sont des doublons techniques stricts.
(Même joueur, même saison, même club -> Erreur d'extraction/jointure)

Exemple de lignes techniques concernées :
                player  season         team
28             Willian    2021      Arsenal
29             Willian    2021      Arsenal
30             Willian    2021      Arsenal
37   Emiliano Martínez    2021  Aston Villa
38   Emiliano Martínez    2021  Aston Villa
119           Jorginho    2021      Chelsea

869 lignes correspondent à des doublons de mercato
(Même joueur, même saison, mais clubs différents -> Transferts de mi-saison)

Exemple de joueurs transférés concernés :
                    player  season     team
0   Ainsley Maitland-Niles    2021  Arsenal
14             Joe Willock    2021  Arsenal
16         Martin Ødegaard    2021  Arsenal
17             Mathew Ryan    2021  Arsenal
25          Sead Kolašinac    2021  Arsenal
26        Shkodran Mustafi    2021  Arsenal


{'doublons_techniques': np.int64(787), 'doublons_mercato': np.int64(869)}

In [6]:
df = fusionner_doublons_techniques(df)

Format initial de la base : (17122, 126)
Format après fusion intelligente des doublons : (16335, 126)


In [7]:
df = fusionner_et_recalculer_mercato(df)

Format avant fusion mercato : (16335, 126)
Format après fusion mercato : (15466, 126)
Recalcul des ratios et statistiques par 90 minutes...
Base de données fusionnée et variables recalculées avec exactitude.



c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\cleaning.py:280: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ).agg(aggregation_rules)
c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\cleaning.py:280: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ).agg(aggregation_rules)


## Homogénéisation des formats

In [8]:
colonnes_dates = ["date_of_birth", "contract_expiration_date"]

# Application ciblée
df = nettoyer_age_et_dates(
    df, colonnes_dates=colonnes_dates
)

Traitement des colonnes de dates : ['date_of_birth', 'contract_expiration_date']
 -> Toutes les heures ont été remises à minuit.
 -> Colonne 'born' (année de naissance) extraite.
Calcul et nettoyage de la colonne 'age'...
 -> Attention : 0 âges manquants remplacés par la médiane (25 ans).
 -> Colonne 'age' convertie strictement en entiers (int).
Nettoyage de l'âge et des dates terminé.



In [9]:
df

,player,season,team,league,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,injury_minor_unknown_nb_d,injury_minor_unknown_nb_m,injury_musculaire,injury_genou,injury_cheville_pied,injury_mollet_tibia,injury_dos_bassin,injury_trauma_severe,injury_medical_repos,injury_minor_unknown
0,Aaron Ciammaglichella,2425,Torino,ITA-Serie A,ITA,MF,19,NaN,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Aaron Connolly,2021,Brighton,ENG-Premier League,IRL,"FW,MF",20,2000.0,17,9,...,29.0,6.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,Aaron Connolly,2122,Brighton,ENG-Premier League,IRL,FW,21,NaN,4,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Aaron Cresswell,2021,West Ham United,ENG-Premier League,ENG,DF,31,1989.0,36,36,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,Aaron Cresswell,2122,West Ham United,ENG-Premier League,ENG,DF,32,1989.0,31,31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15461,Šime Vrsaljko,2021,Atlético Madrid,ESP-La Liga,CRO,MF,28,1992.0,9,6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15462,Šime Vrsaljko,2122,Atlético Madrid,ESP-La Liga,CRO,"DF,MF",29,1992.0,21,10,...,193.0,10.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
15463,Ștefan Radu,2021,Lazio,ITA-Serie A,ROU,DF,34,1986.0,31,30,...,24.0,5.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
15464,Ștefan Radu,2122,Lazio,ITA-Serie A,ROU,DF,35,1986.0,10,6,...,36.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0


## Traitement des variables avec beaucoup de valeurs manquantes

In [10]:
diagnostiquer_valeurs_manquantes(df, seuil=0.01)

Diagnostic des valeurs manquantes (Seuil > 1%)
   • Penalty Kicks_Save% : 94.6% de valeurs manquantes
   • Performance_CS% : 92.8% de valeurs manquantes
   • Performance_Save% : 92.6% de valeurs manquantes
   • Standard_G/SoT : 27.9% de valeurs manquantes
   • contract_expiration_date : 25.2% de valeurs manquantes
   • foot : 17.2% de valeurs manquantes
   • date_of_birth : 17.1% de valeurs manquantes
   • born : 17.1% de valeurs manquantes
   • tm_dob_key : 17.1% de valeurs manquantes
   • sub_position : 17.1% de valeurs manquantes
   • tm_join_key_full : 17.1% de valeurs manquantes
   • date : 17.1% de valeurs manquantes
   • market_value_in_eur : 17.1% de valeurs manquantes
   • name : 17.1% de valeurs manquantes
   • tm_join_key : 17.1% de valeurs manquantes
   • valuation_season_year : 17.1% de valeurs manquantes
   • player_id : 17.1% de valeurs manquantes
   • position : 17.1% de valeurs manquantes
   • Standard_SoT% : 16.9% de valeurs manquantes
   • Standard_G/Sh : 16.9% de va

In [11]:
# Application de la fonction
df = nettoyer_valeurs_manquantes_ciblees(df)

Début du traitement ciblé des valeurs manquantes...
 -> 11 colonnes de performance nettoyées (NaN -> 0).
 -> Propagation inter-saisons terminée pour 9 colonnes fixes.
Finitions terminées (Derniers NaN résiduels convertis en valeurs neutres).


In [12]:
diagnostiquer_valeurs_manquantes(df, seuil=0.01)

Diagnostic des valeurs manquantes (Seuil > 1%)
   • contract_expiration_date : 25.2% de valeurs manquantes
   • born : 17.1% de valeurs manquantes
   • valuation_season_year : 17.1% de valeurs manquantes
   • market_value_in_eur : 17.1% de valeurs manquantes
   • date : 17.1% de valeurs manquantes

Total : 5 colonnes dépassent le seuil de 1%.


In [13]:
df

,player,season,team,league,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,injury_minor_unknown_nb_d,injury_minor_unknown_nb_m,injury_musculaire,injury_genou,injury_cheville_pied,injury_mollet_tibia,injury_dos_bassin,injury_trauma_severe,injury_medical_repos,injury_minor_unknown
0,Aaron Ciammaglichella,2425,Torino,ITA-Serie A,ITA,MF,19,NaN,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Aaron Connolly,2021,Brighton,ENG-Premier League,IRL,"FW,MF",20,2000.0,17,9,...,29.0,6.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,Aaron Connolly,2122,Brighton,ENG-Premier League,IRL,FW,21,NaN,4,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Aaron Cresswell,2021,West Ham United,ENG-Premier League,ENG,DF,31,1989.0,36,36,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,Aaron Cresswell,2122,West Ham United,ENG-Premier League,ENG,DF,32,1989.0,31,31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15461,Šime Vrsaljko,2021,Atlético Madrid,ESP-La Liga,CRO,MF,28,1992.0,9,6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15462,Šime Vrsaljko,2122,Atlético Madrid,ESP-La Liga,CRO,"DF,MF",29,1992.0,21,10,...,193.0,10.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
15463,Ștefan Radu,2021,Lazio,ITA-Serie A,ROU,DF,34,1986.0,31,30,...,24.0,5.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
15464,Ștefan Radu,2122,Lazio,ITA-Serie A,ROU,DF,35,1986.0,10,6,...,36.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0


## Encodage de variables

In [14]:
# Analyser la base des gardiens
var_categorielles = lister_variables_categorielles(df)

Variables catégorielles du dataset :
Liste des variables catégorielles détectées :
   • player (5511 modalités uniques)
   • team (137 modalités uniques)
   • league (5 modalités uniques)
   • nation (129 modalités uniques)
   • pos (10 modalités uniques)
   • join_key (5509 modalités uniques)
   • match_method (17 modalités uniques)
   • date (254 modalités uniques)
   • name (4547 modalités uniques)
   • tm_join_key (4546 modalités uniques)
   • tm_join_key_full (4546 modalités uniques)
   • sub_position (13 modalités uniques)
   • position (5 modalités uniques)
   • foot (3 modalités uniques)

Total : 14 variables catégorielles trouvées.


In [15]:
# Variables catégorielles à traiter
mes_variables = ["pos", "sub_position", "nation", "league", "foot"]

# Lancement de l'encodage
df = encoder_dataset_football(
    df=df,
    colonnes_categoriques=mes_variables,
    df_fifa_historique=df_fifa,
)

Format initial avant encodage : (15466, 126)
Encodage de 'nation' en 10 colonnes binaires Top FIFA (par saison)...
   • Les 10 colonnes classement_FIFA_X ont été injectées.
Profil Joueurs de champ détecté : Encodage Multi-Label de 'pos'.
Encodage One-Hot des colonnes : ['sub_position', 'league', 'foot']
Format final après encodage : (15466, 157)



## Jours contrat restants

In [16]:
df = calculer_jours_contrat_restants(df)

Début du calcul de la durée restante des contrats...
Contrats déjà expirés (valeur négative) : 0
Contrats manquants (NaN)                  : 3897

Statistiques descriptives de la variable calculée :
count    11569.000000
mean      1474.809664
std        721.212718
min          0.000000
25%       1095.000000
50%       1461.000000
75%       1827.000000
max       5113.000000
Calcul terminé avec succès.


In [17]:
df

,player,season,team,nation,age,born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,...,sub_position_Second Striker,league_ENG-Premier League,league_ESP-La Liga,league_FRA-Ligue 1,league_GER-Bundesliga,league_ITA-Serie A,foot_both,foot_left,foot_right,contrat_jours_restants
0,Aaron Ciammaglichella,2425,Torino,ITA,19,NaN,1,0,1,0.0,...,0,0,0,0,0,1,0,0,1,NaN
1,Aaron Connolly,2021,Brighton,IRL,20,2000.0,17,9,791,8.8,...,0,1,0,0,0,0,0,0,1,1096.0
2,Aaron Connolly,2122,Brighton,IRL,21,NaN,4,1,156,1.7,...,0,1,0,0,0,0,0,0,1,NaN
3,Aaron Cresswell,2021,West Ham United,ENG,31,1989.0,36,36,3170,35.2,...,0,1,0,0,0,0,0,1,0,1826.0
4,Aaron Cresswell,2122,West Ham United,ENG,32,1989.0,31,31,2726,30.3,...,0,1,0,0,0,0,0,1,0,1461.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15461,Šime Vrsaljko,2021,Atlético Madrid,CRO,28,1992.0,9,6,519,5.8,...,0,0,1,0,0,0,0,0,1,NaN
15462,Šime Vrsaljko,2122,Atlético Madrid,CRO,29,1992.0,21,10,914,10.2,...,0,0,1,0,0,0,0,0,1,NaN
15463,Ștefan Radu,2021,Lazio,ROU,34,1986.0,31,30,2458,27.3,...,0,0,0,0,0,1,0,1,0,NaN
15464,Ștefan Radu,2122,Lazio,ROU,35,1986.0,10,6,556,6.2,...,0,0,0,0,0,1,0,1,0,NaN


## Suppression de colonnes en double

In [18]:
colonnes_redondantes = [
    "Starts_Starts", "Standard_PK", "Standard_PKatt", "Standard_Gls", "90s", "Playing Time_Min%",
    "Performance_SoTA", "Performance_G+A", "Team Success_+/-", "Team Success_+/-90", "Playing Time_Min", 
    "Penalty Kicks_PKatt", "born", "np_xg", "xg_chain", "Per 90 Minutes_G+A-PK", "Per 90 Minutes_G-PK",
    "join_key", "tm_join_key", "tm_join_key_full", "tm_id", "player_id", "dob_key", "tm_dob_key",
    "match_method", "name", "date_of_birth", "dob_year", "date", "season", "valuation_season_year",
    "contract_expiration_date", "Starts_Mn/Start"
]

df = supprimer_colonnes_du_dataset(df, colonnes_redondantes)

32 colonne(s) supprimée(s) : ['Starts_Starts', 'Standard_PK', 'Standard_PKatt', 'Standard_Gls', '90s', 'Playing Time_Min%', 'Performance_SoTA', 'Performance_G+A', 'Team Success_+/-', 'Team Success_+/-90', 'Playing Time_Min', 'Penalty Kicks_PKatt', 'born', 'np_xg', 'xg_chain', 'Per 90 Minutes_G+A-PK', 'Per 90 Minutes_G-PK', 'join_key', 'tm_join_key', 'tm_join_key_full', 'tm_id', 'player_id', 'dob_key', 'tm_dob_key', 'match_method', 'name', 'date_of_birth', 'date', 'season', 'valuation_season_year', 'contract_expiration_date', 'Starts_Mn/Start']


In [19]:
df

,player,team,nation,age,Playing Time_MP,Playing Time_Starts,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G-PK,...,sub_position_Second Striker,league_ENG-Premier League,league_ESP-La Liga,league_FRA-Ligue 1,league_GER-Bundesliga,league_ITA-Serie A,foot_both,foot_left,foot_right,contrat_jours_restants
0,Aaron Ciammaglichella,Torino,ITA,19,1,0,0.0,0,0,0,...,0,0,0,0,0,1,0,0,1,NaN
1,Aaron Connolly,Brighton,IRL,20,17,9,8.8,2,1,2,...,0,1,0,0,0,0,0,0,1,1096.0
2,Aaron Connolly,Brighton,IRL,21,4,1,1.7,0,0,0,...,0,1,0,0,0,0,0,0,1,NaN
3,Aaron Cresswell,West Ham United,ENG,31,36,36,35.2,0,8,0,...,0,1,0,0,0,0,0,1,0,1826.0
4,Aaron Cresswell,West Ham United,ENG,32,31,31,30.3,2,3,2,...,0,1,0,0,0,0,0,1,0,1461.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15461,Šime Vrsaljko,Atlético Madrid,CRO,28,9,6,5.8,0,0,0,...,0,0,1,0,0,0,0,0,1,NaN
15462,Šime Vrsaljko,Atlético Madrid,CRO,29,21,10,10.2,1,2,1,...,0,0,1,0,0,0,0,0,1,NaN
15463,Ștefan Radu,Lazio,ROU,34,31,30,27.3,0,3,0,...,0,0,0,0,0,1,0,1,0,NaN
15464,Ștefan Radu,Lazio,ROU,35,10,6,6.2,0,0,0,...,0,0,0,0,0,1,0,1,0,NaN


## Traitement des outliers et normalisations

In [20]:
dossier_sortie = r"..\data_finale"

cols_a_normaliser = [
    "market_value_in_eur",
    "Playing Time_MP",
    "Playing Time_Starts",
    "Starts_Compl",
    "Subs_Subs",
    "Subs_Mn/Sub",
    "Subs_unSub",
    "Performance_Gls",
    "Performance_Ast",
    "Performance_PK",
    "Performance_PKatt",
    "Performance_Saves",
    "Performance_Save%",
    "Performance_CS",
    "Performance_CS%",
    "Standard_Sh",
    "Standard_SoT",
    "Standard_SoT%",
    "Standard_G/Sh",
    "Standard_G/SoT",
    "Team Success_PPM",
    "xg",
    "xa",
    "xg_buildup",
]

In [21]:
df_train, df_val, df_test = executer_pipeline_preprocessing(
    df = df,
    dossier_sortie=dossier_sortie,
    cols_a_normaliser=cols_a_normaliser,
    height_min=155,
    height_max=210,
)

Split effectué. Train: 10729 | Val: 2656 | Test: 2081
Outliers traités (Bornes: [155, 210] | Remplacement par la médiane du poste du Train).
Normalisation MinMax appliquée avec succès.
Pipeline terminé ! Fichiers sauvegardés dans : ..\data_finale



In [ ]:
df